[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C27_Model_Compression_Course/04_distillation/04_distillation.ipynb)

# 04 · 知识蒸馏（用 numpy 从零实现）

把大模型的「看法」装进小学生，**从零实现温度蒸馏 + KL，并验证学生确实学到暗知识**。

**路线**：
1. 温度软化 softmax：暗知识浮现（熵随 T 增大）
2. KL 蒸馏损失 + T² 缩放：为什么需要 T²
3. 玩具教师→学生：GD 让学生匹配教师软标签
4. 暗知识有用：软标签蒸馏 vs 只用硬标签
5. feature 蒸馏：匹配中间表示（含维度投影）
6. 完整损失：软标签 KL + 硬标签 CE 加权
7. ✏️ 练习（KL 蒸馏 loss / 温度 / feature 蒸馏 / 效果）
8. 📖 答案 · 🧪 真实 logit 分布胶囊

> **本课纪律**：每个机制都 assert 验证。温度增熵、T² 稳梯度、蒸馏后学生匹配教师——都要跑出来。

## 1 · 温度软化 softmax：暗知识浮现

`softmax(z/T)`，`T>1` 软化分布、放大非目标类的相对概率（暗知识）。
验证：温度越高，分布的熵越大（越软）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def softmax(z, T=1.0, axis=-1):
    z = np.asarray(z, float) / T
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def entropy(p, axis=-1):
    return -(p * np.log(p + 1e-12)).sum(axis=axis)

z = np.array([5.0, 2.0, 1.0, 0.5])               # 一个自信的教师 logit
print(f"{'T':>4} {'probs':<32} {'熵':>6}")
ents = []
for T in [1, 2, 4, 8]:
    p = softmax(z, T); H = entropy(p)
    ents.append(H)
    print(f'{T:>4} {str(np.round(p,3)):<32} {H:>6.3f}')
assert all(ents[i] < ents[i+1] for i in range(len(ents)-1)), '温度越高熵越大'
print('✅ 温度越高分布越软、熵越大 —— 非目标类的相对关系(暗知识)被放大')

## 2 · KL 蒸馏损失 + T² 缩放

蒸馏损失 = `T² · KL(p_teacher || p_student)`，都在温度 T 下算软标签。
**为什么 T²**：软标签对学生 logit 的梯度带 `1/T²` 因子，乘 T² 抵消，让梯度量级与温度无关。

In [ ]:
def kl_div(p, q, axis=-1):
    return (p * (np.log(p + 1e-12) - np.log(q + 1e-12))).sum(axis=axis).mean()

def kd_loss(teacher_logits, student_logits, T, use_t2=True):
    p_t = softmax(teacher_logits, T)
    p_s = softmax(student_logits, T)
    loss = kl_div(p_t, p_s)
    return loss * (T*T if use_t2 else 1.0)

def kd_grad_magnitude(teacher_logits, student_logits, T, use_t2=True):
    '''KD loss 对 student_logits 的梯度量级。grad = (p_s - p_t)/T, T² 时再 ×T²。'''
    p_t = softmax(teacher_logits, T); p_s = softmax(student_logits, T)
    g = (p_s - p_t) / T / p_t.shape[0]
    if use_t2: g = g * (T*T)
    return np.abs(g).mean()

C, N = 10, 64
tl = rng.standard_normal((N, C)) * 3
sl = rng.standard_normal((N, C)) * 3
print(f"{'T':>4} {'梯度量级(无T²)':>16} {'梯度量级(有T²)':>16}")
mags_no, mags_yes = [], []
for T in [1, 2, 4, 8]:
    gn = kd_grad_magnitude(tl, sl, T, use_t2=False)
    gy = kd_grad_magnitude(tl, sl, T, use_t2=True)
    mags_no.append(gn); mags_yes.append(gy)
    print(f'{T:>4} {gn:>16.2e} {gy:>16.2e}')
# 无 T²: 梯度随 T 暴跌; 有 T²: 量级稳定
assert mags_no[0] / mags_no[-1] > 5, '无 T²: 梯度应随 T 大幅下降'
assert max(mags_yes) / min(mags_yes) < 3, '有 T²: 梯度量级应稳定'
print('✅ T² 缩放让蒸馏梯度量级与温度无关 —— 不加 T² 软标签项会随 T 失效')

## 3 · 玩具教师→学生：学生匹配教师软标签

用梯度下降在学生 logit 上最小化 KD 损失，验证学生分布**收敛到教师分布**。
KD 对学生 logit 的梯度（每行）：`(p_s - p_t)/T`（乘 T² 后是 `(p_s-p_t)·T`）。

In [ ]:
C, N = 10, 32
teacher_logits = rng.standard_normal((N, C)) * 3      # 固定教师
student_logits = rng.standard_normal((N, C)) * 3      # 学生(待训练)
T = 4.0; lr = 4.0
p_t = softmax(teacher_logits, T)

loss0 = kd_loss(teacher_logits, student_logits, T)
sl = student_logits.copy()
for step in range(500):                                # GD on student logits
    p_s = softmax(sl, T)
    grad = (p_s - p_t) / T * (T*T) / N                 # KD(含T²) 对 logit 的梯度(mean loss -> /N)
    sl -= lr * grad
loss1 = kd_loss(teacher_logits, sl, T)
print(f'KD 损失: 初始 {loss0:.4f} -> 训练后 {loss1:.6f}')
print(f'学生分布是否匹配教师(T下): {np.allclose(softmax(sl,T), p_t, atol=0.05)}')
assert loss1 < loss0 / 10, 'KD 损失应大幅下降'
assert np.allclose(softmax(sl, T), p_t, atol=0.05), '学生软标签应收敛到教师'
print('✅ GD 最小化 KD 损失，学生软标签收敛到教师 —— 学生学到了教师的完整分布')

## 4 · 暗知识有用：软标签 vs 只用硬标签

软标签携带类间结构(暗知识)，硬标签没有。验证：让学生匹配教师软标签，
学到的分布比只匹配硬标签(one-hot)更接近教师在**非目标类**上的相对关系。

In [ ]:
C, N = 6, 40
teacher_logits = rng.standard_normal((N, C)) * 2.5
p_t = softmax(teacher_logits, 1.0)
labels = p_t.argmax(axis=1)                            # 硬标签=教师的 top1
onehot = np.eye(C)[labels]

# 学生A: 学软标签(暗知识)   学生B: 只学硬标签(one-hot)
def train_student(target, T, steps=1500, lr=4.0):
    sl = rng.standard_normal((N, C)) * 2.5
    for _ in range(steps):
        p_s = softmax(sl, T)
        sl -= lr * (p_s - target) / N                  # 对 target 的交叉熵/KL 梯度
    return softmax(sl, 1.0)

stu_soft = train_student(softmax(teacher_logits, 4.0), T=4.0)   # 学软标签
stu_hard = train_student(onehot, T=1.0)                          # 学硬标签

# 衡量: 在非目标类上, 谁更接近教师的相对结构(用 KL 到教师 T=1 分布)
kl_soft = kl_div(p_t, stu_soft)
kl_hard = kl_div(p_t, stu_hard)
print(f'学软标签的学生 到教师的 KL = {kl_soft:.4f}')
print(f'学硬标签的学生 到教师的 KL = {kl_hard:.4f}')
assert kl_soft < kl_hard, '学软标签应更接近教师(学到暗知识)'
print('✅ 软标签学生更接近教师的完整分布 —— 暗知识(类间结构)确实被传递了')

## 5 · feature 蒸馏：匹配中间表示

知识不只在输出。feature 蒸馏让学生中间层拟合教师中间层。
学生维度通常小于教师，需一个**投影**对齐维度，再最小化 MSE。

In [ ]:
d_teacher, d_student, N = 64, 32, 50
teacher_feat = rng.standard_normal((N, d_teacher))    # 教师中间表示
student_feat = rng.standard_normal((N, d_student))    # 学生中间表示(维度更小)

# 投影: 把学生特征(d_student) 投到教师维度(d_teacher) 再比 MSE
def feature_distill_loss(student_feat, teacher_feat, W_proj):
    student_projected = student_feat @ W_proj          # (N, d_teacher)
    return ((student_projected - teacher_feat) ** 2).mean()

# 训练投影 + 学生特征去匹配教师(这里固定教师, 学投影与学生特征)
W_proj = rng.standard_normal((d_student, d_teacher)) * 0.1
sf = student_feat.copy(); lr = 0.01
loss0 = feature_distill_loss(sf, teacher_feat, W_proj)
for _ in range(500):
    sp = sf @ W_proj
    diff = (sp - teacher_feat) / N
    W_proj -= lr * (sf.T @ diff)                       # dL/dW_proj
    sf -= lr * (diff @ W_proj.T)                       # dL/d student_feat
loss1 = feature_distill_loss(sf, teacher_feat, W_proj)
print(f'feature 蒸馏 MSE: 初始 {loss0:.4f} -> 训练后 {loss1:.4f}')
assert loss1 < loss0 / 2, 'feature 蒸馏应降低 MSE'
print('✅ 通过投影对齐维度，学生中间表示拟合教师 —— feature 蒸馏生效')

## 6 · 完整损失：软标签 KL + 硬标签 CE

真实蒸馏损失 = `α·T²·KL(教师||学生) + (1-α)·CE(真值, 学生)`。
软标签项学暗知识、硬标签项学真值(防教师犯错)。验证组合损失能同时降两项。

In [ ]:
def cross_entropy(labels, logits):
    p = softmax(logits, 1.0)
    return -np.log(p[np.arange(len(labels)), labels] + 1e-12).mean()

def combined_loss(teacher_logits, student_logits, labels, T, alpha):
    kd = kd_loss(teacher_logits, student_logits, T)            # 软标签(含 T²)
    ce = cross_entropy(labels, student_logits)                 # 硬标签
    return alpha * kd + (1 - alpha) * ce, kd, ce

C, N = 8, 48
teacher_logits = rng.standard_normal((N, C)) * 3
labels = rng.integers(0, C, N)                                 # 真值(可能 != 教师top1)
sl = rng.standard_normal((N, C)) * 3
T, alpha, lr = 4.0, 0.7, 4.0
p_t = softmax(teacher_logits, T)
tot0, kd0, ce0 = combined_loss(teacher_logits, sl, labels, T, alpha)
for _ in range(800):
    p_s_T = softmax(sl, T); p_s_1 = softmax(sl, 1.0)
    grad_kd = alpha * (p_s_T - p_t) / T * (T*T) / N
    grad_ce = (1-alpha) * (p_s_1 - np.eye(C)[labels]) / N
    sl -= lr * (grad_kd + grad_ce)
tot1, kd1, ce1 = combined_loss(teacher_logits, sl, labels, T, alpha)
print(f'总损失: {tot0:.4f} -> {tot1:.4f}  (KD {kd0:.3f}->{kd1:.3f}, CE {ce0:.3f}->{ce1:.3f})')
assert tot1 < tot0, '组合损失应下降'
assert kd1 < kd0 and ce1 < ce0, '软标签项与硬标签项都应改善'
print('✅ 组合损失同时降低 KD(学教师) 与 CE(学真值) —— 这是标准蒸馏配方')

---
## ✏️ 练习 1：KL 蒸馏损失

实现 `my_kd_loss(teacher_logits, student_logits, T)`：返回 `T²·KL(p_teacher||p_student)`，软标签都在温度 T 下算。
目标：学生越接近教师，损失越小；学生==教师时损失≈0。

In [ ]:
def my_kd_loss(teacher_logits, student_logits, T):
    # TODO:
    #   p_t = softmax(teacher_logits, T); p_s = softmax(student_logits, T)
    #   kl = mean over rows of sum_i p_t*(log p_t - log p_s)
    #   返回 kl * T*T
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
tl = rng.standard_normal((20, 10)) * 3
assert my_kd_loss(tl, tl.copy(), 4.0) < 1e-9, '学生==教师时 KD 损失应≈0'
sl_far = rng.standard_normal((20, 10)) * 3
sl_near = tl + rng.standard_normal((20, 10)) * 0.1     # 接近教师
assert my_kd_loss(tl, sl_near, 4.0) < my_kd_loss(tl, sl_far, 4.0), '越近损失越小'
print('✅ 练习 1 通过：KD 损失正确（学生==教师→0，越近越小）')

## ✏️ 练习 2：温度软化

实现 `soften_and_entropy(logits, T)`：返回温度 T 下的 softmax 分布和它的熵。
目标：T 越大，熵越大（分布越软）。

In [ ]:
def soften_and_entropy(logits, T):
    # TODO: p = softmax(logits, T); H = -sum(p*log p) (每行, 取均值)
    #       返回 (p, H_mean)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
logits = rng.standard_normal((16, 8)) * 4
Hs = [soften_and_entropy(logits, T)[1] for T in [1, 2, 4, 8]]
assert all(Hs[i] < Hs[i+1] for i in range(len(Hs)-1)), '温度越高熵越大'
p1, _ = soften_and_entropy(logits, 1.0)
assert np.allclose(p1.sum(axis=1), 1.0), '应是合法概率分布'
print(f'✅ 练习 2 通过：熵随温度递增 {[round(h,2) for h in Hs]}')

## ✏️ 练习 3：feature 蒸馏损失

实现 `feat_distill(student_feat, teacher_feat, W_proj)`：把学生特征经投影 `W_proj` 对齐到教师维度，返回 MSE。
目标：维度对齐后能算出标量 MSE。

In [ ]:
def feat_distill(student_feat, teacher_feat, W_proj):
    # TODO: student_projected = student_feat @ W_proj; 返回 mean((proj - teacher)^2)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
sf = rng.standard_normal((30, 16)); tf = rng.standard_normal((30, 48))
Wp = rng.standard_normal((16, 48)) * 0.1
loss = feat_distill(sf, tf, Wp)
assert np.isscalar(loss) or loss.ndim == 0, '应返回标量 MSE'
# 完美投影(教师就是学生的某个线性变换)时 MSE 应≈0
Wp_perfect = np.linalg.lstsq(sf, tf, rcond=None)[0]
assert feat_distill(sf, tf, Wp_perfect) < loss, '更好的投影应给更小 MSE'
print('✅ 练习 3 通过：feature 蒸馏 MSE 正确，投影对齐维度')

## ✏️ 练习 4：蒸馏效果——软标签 vs 硬标签

实现 `distill_gap(teacher_logits, T, steps)`：分别训一个学软标签、一个学硬标签的学生，
返回 `(kl_soft, kl_hard)` 各自到教师分布的 KL。目标：`kl_soft < kl_hard`（暗知识有用）。

In [ ]:
def distill_gap(teacher_logits, T=4.0, steps=400, lr=0.5):
    # TODO: p_t1=softmax(teacher,1); labels=argmax; onehot=eye[labels]
    #   学生A 学 softmax(teacher,T)(软); 学生B 学 onehot(硬)
    #   各训 steps 步 GD, 返回 (kl到教师 of A, kl到教师 of B)  (都用 T=1 分布比)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
tl = rng.standard_normal((40, 6)) * 2.5
kl_soft, kl_hard = distill_gap(tl, T=4.0)
print(f'学软标签 KL={kl_soft:.4f}   学硬标签 KL={kl_hard:.4f}')
assert kl_soft < kl_hard, '软标签(暗知识)应让学生更接近教师'
print('✅ 练习 4 通过：软标签学生更接近教师 —— 暗知识被传递')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_kd_loss(teacher_logits, student_logits, T):
    p_t = softmax(teacher_logits, T); p_s = softmax(student_logits, T)
    kl = (p_t * (np.log(p_t + 1e-12) - np.log(p_s + 1e-12))).sum(axis=-1).mean()
    return kl * T * T

In [ ]:
# 练习 2 参考答案
def soften_and_entropy(logits, T):
    p = softmax(logits, T)
    H = -(p * np.log(p + 1e-12)).sum(axis=-1).mean()
    return p, H

In [ ]:
# 练习 3 参考答案
def feat_distill(student_feat, teacher_feat, W_proj):
    return ((student_feat @ W_proj - teacher_feat) ** 2).mean()

In [ ]:
# 练习 4 参考答案
def distill_gap(teacher_logits, T=4.0, steps=1500, lr=4.0):
    N, C = teacher_logits.shape
    p_t1 = softmax(teacher_logits, 1.0)
    labels = p_t1.argmax(axis=1); onehot = np.eye(C)[labels]
    def train(target, Tt):
        sl = rng.standard_normal((N, C)) * 2.5
        for _ in range(steps):
            p_s = softmax(sl, Tt)
            sl -= lr * (p_s - target) / N
        return softmax(sl, 1.0)
    a = train(softmax(teacher_logits, T), T)
    b = train(onehot, 1.0)
    kld = lambda q: (p_t1 * (np.log(p_t1+1e-12) - np.log(q+1e-12))).sum(-1).mean()
    return kld(a), kld(b)

---
## 🧪 真实数据胶囊：真实 GPT-2 的 next-token 分布做教师

用**真实 GPT-2** 对一句话的 next-token logit 当教师软标签，训一个小学生匹配它，看暗知识。
**联网失败自动回退**到统计匹配的合成 logit（自信、长尾），结论不变。

In [ ]:
def load_gpt2_logits():
    '''真实 GPT-2 对一句话最后一个位置的 next-token logits(取前 50 个高频 token);
       失败回退合成。'''
    try:
        import torch
        from transformers import GPT2LMHeadModel, GPT2Tokenizer
        tok = GPT2Tokenizer.from_pretrained('gpt2')
        m = GPT2LMHeadModel.from_pretrained('gpt2').eval()
        ids = tok('The capital of France is', return_tensors='pt')
        with torch.no_grad():
            logits = m(**ids).logits[0, -1]
        top = logits.topk(50).values.detach().numpy().astype(np.float64)
        print(f'[真实 GPT-2] next-token top-50 logits, max={top.max():.2f}')
        return top
    except Exception as e:
        print(f'[回退合成] ({type(e).__name__}) 用统计匹配合成 logits')
        rng2 = np.random.default_rng(13)
        # 自信分布: 一个大峰 + 长尾(暗知识)
        t = np.sort(rng2.standard_normal(50))[::-1] * 2
        t[0] += 6.0                                     # 一个明显的 top token
        return t

teacher_logits = load_gpt2_logits()[None, :]            # (1, 50)
p_t1 = softmax(teacher_logits, 1.0)
print(f'教师 top token 概率={p_t1.max():.3f}  熵(T=1)={entropy(p_t1[0]):.3f}')
print(f'教师 熵(T=4)={entropy(softmax(teacher_logits,4.0)[0]):.3f}  <- 温度放大了暗知识')

In [ ]:
def distill_to_student(teacher_logits, T=4.0, steps=500, lr=0.5):
    # TODO: 随机初始化学生 logits(同形状), GD 最小化 my_kd_loss, 返回训练后的学生 logits
    raise NotImplementedError

In [ ]:
# 自测
student = distill_to_student(teacher_logits, T=4.0)
kd_final = my_kd_loss(teacher_logits, student, 4.0)
match = np.allclose(softmax(student, 4.0), softmax(teacher_logits, 4.0), atol=0.05)
print(f'蒸馏后 KD 损失 = {kd_final:.5f}')
print(f'学生 next-token 分布匹配教师(T下)? {match}')
assert kd_final < 0.1, 'KD 损失应降到很小'
assert match, '学生应学到教师的完整分布(含长尾暗知识)'
print('✅ 胶囊通过：小学生学到真实 GPT-2 的 next-token 分布(含暗知识) —— 这就是 LLM 蒸馏的核心')

In [ ]:
# 📖 胶囊参考答案
def distill_to_student(teacher_logits, T=4.0, steps=2000, lr=4.0):
    p_t = softmax(teacher_logits, T)
    sl = rng.standard_normal(teacher_logits.shape) * 2
    N = teacher_logits.shape[0]
    for _ in range(steps):
        p_s = softmax(sl, T)
        sl -= lr * (p_s - p_t) / T * (T*T) / N
    return sl

### 小结
- **蒸馏 = 把大教师的「看法」装进小学生**：学生模仿教师输出，比只学硬标签信息更丰富。
- **暗知识**：软标签的非目标类相对概率编码了类间相似度(『猫像狗不像车』)，硬标签没有。
- **温度 T>1** 软化分布、放大暗知识(熵增)；教师学生用同一 T 训练，学生推理用 T=1。
- **蒸馏损失 = T²·KL(教师||学生)**；T² 抵消软标签梯度的 1/T² 因子，让量级与温度无关(**别忘 T²**)。
- **完整损失** = α·KD(软标签) + (1-α)·CE(硬标签)：学教师 + 防教师犯错。
- 蒸馏什么：**logit**(最简)/**feature**(中间层,需投影)/**attention**(Transformer)/**自蒸馏**(提质)；可组合。
- 蒸馏**不受 bit 限制**(直接换小模型)、与量化/剪枝**正交可叠加**；代价是要训练 + 需好教师。

下一站：**模块 05 · 剪枝与稀疏** —— 不换模型，而是把既有模型的权重删成 0。